# BIOT 6900 · Module 3 Lab, Part 2: Real Proteins

**Self-paced · not graded · runs offline from the Module 3 data pack**

In Part 1 you ran five steps on a toy model. Now you run the **same steps on real human proteins**: pre-computed ESM C embeddings and real Open Targets tractability labels. The Module 3 Lab Guide (Part 2, Steps 2.1–2.9) walks through each cell, so you can work through it on your own.

| Step | What you do |
|---|---|
| 2.1 | Load the data pack |
| 2.2 | Understand the labels |
| 2.3 | Map the embedding space and place p53 |
| 2.4 | Nearest neighbours |
| 2.5 | Train and validate the tractability prior (random vs family split) |
| 2.6 | Where does p53 land? |
| 2.7 | ESM C versus ESM-2 |
| 2.8 | Variant scores for p53 |
| 2.9 | Check-in questions |

**Before you start:** unzip the Module 3 data pack into a folder called `data/` next to this notebook, and install scikit-learn and matplotlib once (`conda install scikit-learn matplotlib` inside `biot6900`). Your exact numbers depend on the data pack, so the guide tells you **what to look for** rather than fixed values.

**Keep this notebook:** the Module 3 assignment reuses its functions.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score

DATA = Path("data")
needed = ["panel.tsv", "panel_emb_main.npy", "panel_emb_baseline.npy", "manifest.json",
          "variant_scores_TP53.tsv", "anchor_variants.tsv"]
missing = [f for f in needed if not (DATA / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing from {DATA.resolve()}: {missing}\n"
                            "Unzip the Module 3 data pack into a folder called data/ next to this notebook.")
SEED = 6900
print("Data pack found.")

## 2.1 · Load the data pack

`panel.tsv` lists the labelled proteins; `panel_emb_main.npy` holds one embedding row per protein, **in the same order**. Embeddings are stored as float16 to keep the download small, so we convert them to float32 straight away.

In [ ]:
panel = pd.read_csv(DATA / "panel.tsv", sep="\t")
X = np.load(DATA / "panel_emb_main.npy").astype("float32")
X_base = np.load(DATA / "panel_emb_baseline.npy").astype("float32")
manifest = json.load(open(DATA / "manifest.json"))

assert len(panel) == len(X) == len(X_base), "table and embeddings are misaligned"
print("Panel:", panel.shape, "| main embeddings:", X.shape, "| baseline embeddings:", X_base.shape)
print("Models:", json.dumps(manifest["embedding_models"], indent=1))
if manifest.get("dry_run"):
    print("\n*** WARNING: this data pack was made in DRY RUN mode (fake data). Numbers are not real biology. ***")

## 2.2 · Understand the labels

Open Targets **clinical precedence**: `sm_clinical` (small molecule), `ab_clinical` (antibody), `any_clinical` (any modality). Only Approved, Advanced Clinical, or Phase 1 evidence counts.

Two things to keep in mind: the panel is **enriched** (about a third positive, far above the natural rate), so read probabilities as rankings; and the "Druggable Family" label was deliberately left out because it is assigned from family membership.

In [ ]:
print("Fraction of panel with clinical precedence:")
print(panel[["sm_clinical", "ab_clinical", "any_clinical"]].mean().round(3).to_string())
print(f"\nFamily groups: {panel.family_group.nunique()} "
      f"({panel.family_group.str.startswith('singleton:').sum()} proteins have no family annotation)")
top_fams = panel.family_group.value_counts().head(8)
print("\nLargest family groups and their clinical-precedence rate:")
print(panel[panel.family_group.isin(top_fams.index)].groupby("family_group")["any_clinical"]
      .agg(n="size", frac_clinical="mean").sort_values("n", ascending=False).round(2))

## 2.3 · Map the embedding space

Squash the embeddings to 2D (PCA), colour the eight largest families, and mark p53. Compare with the toy plot from Part 1.

In [ ]:
Xc = X - X.mean(0)
_, _, Vt = np.linalg.svd(Xc, full_matrices=False)
xy = Xc @ Vt[:2].T
panel["pc1"], panel["pc2"] = xy[:, 0], xy[:, 1]

fig, ax = plt.subplots(figsize=(9, 5.5))
other = ~panel.family_group.isin(top_fams.index)
ax.scatter(panel.pc1[other], panel.pc2[other], s=8, c="#D5DCE2", label="other families")
for fam in top_fams.index:
    g = panel[panel.family_group == fam]
    ax.scatter(g.pc1, g.pc2, s=14, label=fam[:40])
if "TP53" in set(panel.gene):
    p = panel[panel.gene == "TP53"].iloc[0]
    ax.scatter(p.pc1, p.pc2, s=160, marker="*", c="#C8553D", edgecolor="k", zorder=5, label="p53")
ax.set_xticks([]); ax.set_yticks([]); ax.set_title("ESM C embeddings of the panel, 2D projection")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1)); plt.tight_layout(); plt.show()

## 2.4 · Nearest neighbours

Cosine similarity: normalise each row to length 1, then one matrix product gives every pairwise similarity. `nearest()` is reused in the assignment.

In [ ]:
def cosine_matrix(A, B=None):
    A = A / np.linalg.norm(A, axis=1, keepdims=True)
    B = A if B is None else B / np.linalg.norm(B, axis=1, keepdims=True)
    return A @ B.T

S = cosine_matrix(X)

def nearest(gene, k=5):
    i = panel.index[panel.gene == gene][0]
    sims = S[i].copy(); sims[i] = -np.inf
    top = np.argsort(-sims)[:k]
    return panel.loc[top, ["gene", "family_group", "any_clinical"]].assign(cosine=sims[top].round(3))

print("Nearest neighbours of TP53:")
print(nearest("TP53").to_string(index=False))

kinases = panel[panel.family_group.str.contains("kinase", case=False)].gene.tolist()
example = kinases[0] if kinases else panel.gene.iloc[0]      # ✎ change to any gene in the panel
print(f"\nNearest neighbours of {example}:")
print(nearest(example).to_string(index=False))

## 2.5 · Train and validate the tractability prior

A scaled logistic regression on `any_clinical`, evaluated two ways with **out-of-fold** predictions (every protein is predicted by a model that never saw it):

* **Random split** · `StratifiedKFold`: relatives can sit on both sides.
* **Family split** · `GroupKFold` with `groups = family_group`: whole families are held out.

Two metrics: **AUROC** (ranking quality; 0.5 = chance) and **average precision** (precision–recall; chance = the positive fraction). `evaluate()` is reused in the assignment.

In [ ]:
def make_model():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=5000))

def evaluate(X, y, groups, folds=5):
    y = np.asarray(y).astype(int)
    rand = cross_val_predict(make_model(), X, y, cv=StratifiedKFold(folds, shuffle=True, random_state=SEED),
                             method="predict_proba")[:, 1]
    fam = cross_val_predict(make_model(), X, y, cv=GroupKFold(folds), groups=groups, method="predict_proba")[:, 1]
    table = pd.DataFrame({
        "AUROC": [roc_auc_score(y, rand), roc_auc_score(y, fam)],
        "Average precision": [average_precision_score(y, rand), average_precision_score(y, fam)],
    }, index=["Random split", "Family split"]).round(3)
    return table, rand, fam

y = panel.any_clinical.values
res_main, oof_rand, oof_fam = evaluate(X, y, panel.family_group)
print(res_main)
print(f"\nChance level for average precision = positive fraction = {y.mean():.3f}")

ax = res_main.T.plot.bar(figsize=(6, 3.2), color=["#1C7293", "#C8553D"], rot=0)
ax.set_ylim(0, 1); ax.set_title("Same model, two ways of splitting"); plt.tight_layout(); plt.show()

**Look for:** the family-split scores are lower. The size of that gap on real proteins is the key result of Part 2. Which number would you report to a discovery team?

## 2.6 · Where does p53 land?

p53's **family-split, out-of-fold** probability: a prediction made by a model that never saw p53 or its family during training.

In [ ]:
panel["p_oof_family"] = oof_fam
if "TP53" in set(panel.gene):
    p53 = panel[panel.gene == "TP53"].iloc[0]
    pct = (panel.p_oof_family < p53.p_oof_family).mean() * 100
    print(f"p53: P(tractable) = {p53.p_oof_family:.2f} | higher than {pct:.0f}% of the panel "
          f"| Open Targets clinical precedence: {bool(p53.any_clinical)}")
print("\nHighest-scoring proteins WITHOUT clinical precedence (possible 'next targets' or false positives):")
print(panel[~panel.any_clinical].nlargest(8, "p_oof_family")[["gene", "family_group", "p_oof_family"]]
      .round(2).to_string(index=False))

**Discuss:** does the prior call p53 tractable? Either answer teaches something: resembling drugged proteins is not the same as having a druggable pocket, and mutant-specific pockets such as Y220C are invisible to a model trained on historical labels.

## 2.7 · ESM C versus ESM-2

Repeat the evaluation with the baseline embeddings (ESM-2 35M, 480 numbers), then check whether the two models rank proteins the same way.

In [ ]:
res_base, _, oof_fam_base = evaluate(X_base, y, panel.family_group)
compare = pd.concat({manifest["embedding_models"]["main"]["model"]: res_main,
                     manifest["embedding_models"]["baseline"]["model"]: res_base})
print(compare)
rho = spearmanr(oof_fam, oof_fam_base).statistic
print(f"\nSpearman correlation of the two models' family-split predictions: {rho:.2f}")

**Look for:** whether the larger model gives a meaningfully better prior, and whether the two rankings agree. Agreement makes a ranking more robust.

## 2.8 · Variant scores for p53

Every possible single substitution in p53, scored as `log P(mutant) − log P(wild-type)` with ESM-2 650M. `percentile_in_protein` says how extreme a variant is relative to all possible changes in p53 (low = more damaging than most).

In [ ]:
v = pd.read_csv(DATA / "variant_scores_TP53.tsv", sep="\t")
anchors = pd.read_csv(DATA / "anchor_variants.tsv", sep="\t")
hot = anchors[anchors.gene == "TP53"]
print(hot[["variant", "note", "score", "percentile_in_protein"]].round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(v.score, bins=60, color="#C9D6DF")
for _, r in hot.iterrows():
    ax.axvline(r.score, color="#C8553D", lw=1)
    ax.text(r.score, ax.get_ylim()[1] * 0.92, r.variant, rotation=90, fontsize=7, ha="right", va="top")
ax.set_xlabel("score = log P(mutant) − log P(wild-type)"); ax.set_ylabel("number of substitutions")
ax.set_title(f"All {len(v):,} possible p53 substitutions, with cancer hotspots marked"); plt.tight_layout(); plt.show()

per_pos = v.groupby("position").score.mean()
print("\nPositions where substitutions are least tolerated (mean score):")
print(per_pos.nsmallest(8).round(2).to_string())

**Look for:** whether the hotspots sit in the damaging (strongly negative) tail, and whether any sit closer to the middle. Why might a cancer driver not be the change evolution disfavours most?

## 2.9 · Check-in (not graded)

Answer briefly in the cell below:

1. How large was the random versus family gap on real proteins, and which number would you report to a discovery team?
2. Did ESM C clearly beat ESM-2? What does that suggest about model size for this task?
3. Name one reason a p53 hotspot might score as only moderately surprising.

**Next:** the Module 3 assignment applies these steps to your own Module 2 targets.

*Your answers:*